In [1]:
import torch
import numpy as np
from PIL import Image
from transformers import AutoModelForVision2Seq
from transformers import AutoProcessor
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(
    "openvla/openvla-7b",
    trust_remote_code=True,
)
print("Processor type:", type(processor).__name__)

vla = AutoModelForVision2Seq.from_pretrained(
    "openvla/openvla-7b",
    attn_implementation="eager",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    quantization_config=bnb_config,
)

print("Model 로드 완료")
print(f"GPU 메모리 사용: {torch.cuda.memory_allocated()/1e9:.2f} GB")

image = Image.fromarray(
    (np.random.rand(224, 224, 3) * 255).astype(np.uint8)
)
instruction = "pick up the can"
prompt = f"In: What action should the robot take to {instruction}?\nOut:"

inputs = processor(prompt, image).to("cuda:0", dtype=torch.float16)
print(f"inputs.keys(): {list(inputs.keys())}")

# attention_mask 는 전달하지 않는다 -- predict_action 이 빈 토큰(29871) 을 input_ids 에만
# 덧붙여 mask 와 길이가 1 어긋나므로 (eager attention 에서 크래시), generate 가 mask 를
# 알아서 생성하게 둔다
with torch.no_grad():
    action = vla.predict_action(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        unnorm_key="bridge_orig",
        do_sample=False,
    )
print(f"Action shape: {action.shape}")
print(f"Action : {action}")
print(f"GPU 메모리 : {torch.cuda.memory_allocated()/1e9:.2f} GB")


/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Processor type: PrismaticProcessor


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading checkpoint shards: 100%|██████████| 3/3 [00:04<00:00,  1.55s/it]
/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model 로드 완료
GPU 메모리 사용: 4.38 GB
inputs.keys(): ['input_ids', 'attention_mask', 'pixel_values']
Action shape: (7,)
Action : [-2.08787322e-04  5.40355426e-03 -2.12870912e-02  5.60846725e-03
 -1.41778920e-02  7.89113943e-02  9.96078431e-01]
GPU 메모리 : 4.39 GB


/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [2]:
import time
import torch
import numpy as np
from PIL import Image
from transformers import AutoModelForVision2Seq
from transformers import AutoProcessor
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
processor = AutoProcessor.from_pretrained("openvla/openvla-7b", trust_remote_code=True)
vla = AutoModelForVision2Seq.from_pretrained(
    "openvla/openvla-7b",
    attn_implementation="eager",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    quantization_config=bnb_config,
)

image = Image.fromarray((np.random.rand(224, 224, 3) * 255).astype(np.uint8))
instruction = "pick up the can"
prompt = f"In: What action should the robot take to {instruction}?\nOut:"
inputs = processor(prompt, image).to("cuda:0", dtype=torch.float16)

for i in range(5):
    with torch.no_grad():
        action = vla.predict_action(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            unnorm_key="bridge_orig", do_sample=False,
        )
print("warm-up 완료")

latencies = []

for i in range(100):
    # 새 이미지 (cache 효과 방지) -- warm-up 과 동일한 3채널 RGB
    image = Image.fromarray((np.random.rand(224, 224, 3) * 255).astype(np.uint8))
    inputs = processor(prompt, image).to("cuda:0", dtype=torch.float16)

    torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad():
        action = vla.predict_action(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            unnorm_key="bridge_orig", do_sample=False,
        )
    torch.cuda.synchronize()
    elapsed_ms = (time.time() - start) * 1000
    latencies.append(elapsed_ms)

    if (i + 1) % 10 == 0:
        print(f"{i+1}/100: latest = {elapsed_ms:.1f} ms")

arr = np.array(latencies)
print("\n[2-4] Latency 통계")
print(f"mean : {arr.mean():.1f} ms")
print(f"median : {np.median(arr):.1f} ms")
print(f"std : {arr.std():.1f} ms")
print(f"min : {arr.min():.1f} ms")
print(f"max : {arr.max():.1f} ms")
print(f"p95 : {np.percentile(arr, 95):.1f} ms")
print(f"p99 : {np.percentile(arr, 99):.1f} ms")
print()
print(f"Throughput : {1000 / arr.mean():.2f} Hz")

np.save("openvla_latency_4070_int4.npy", arr)
print("\n 결과 저장: openvla_latency_4070_int4.npy")
print("-> Phase 7 의 산출물 v3 에서 비교 baseline 으로 사용")

/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading checkpoint shards: 100%|██████████| 3/3 [00:05<00:00,  1.93s/it]
/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_obl

warm-up 완료
10/100: latest = 297.9 ms
20/100: latest = 293.2 ms
30/100: latest = 293.4 ms
40/100: latest = 292.1 ms
50/100: latest = 293.9 ms
60/100: latest = 294.6 ms
70/100: latest = 295.5 ms
80/100: latest = 294.2 ms
90/100: latest = 294.9 ms
100/100: latest = 294.9 ms

[2-4] Latency 통계
mean : 300.3 ms
median : 301.3 ms
std : 3.8 ms
min : 290.4 ms
max : 308.2 ms
p95 : 304.8 ms
p99 : 305.6 ms

Throughput : 3.33 Hz

 결과 저장: openvla_latency_4070_int4.npy
-> Phase 7 의 산출물 v3 에서 비교 baseline 으로 사용


In [1]:
"""OpenVLA int4 메모리 상세 실측 (findings.md §4.1 Block 1 / methodology.md §4).

3개 시점에서 torch allocator 값과 nvidia-smi 디바이스 사용량 증가분을 함께 읽는다:
    A. CUDA context 초기화 직후 (텐서 1개만) -- context 단독 크기
    B. 모델 로드 직후                        -- 가중치 footprint (2026-06 값 4.38 GB 대응)
    C. predict_action 1회 후                 -- activation·KV cache 포함 peak

nvidia-smi 값은 프로세스 행 매칭이 아니라 디바이스 전체 사용량의 baseline 대비
증가분(delta)이다 -- Docker 컨테이너에서는 nvidia-smi 가 보여주는 PID 가 호스트
네임스페이스 값이라 os.getpid() 와 매칭이 구조적으로 불가능하기 때문.
(전제: 측정 중 다른 프로세스의 GPU 사용량이 일정)

로드 조건은 2026-06 실측 (practice.ipynb) 과 동일: nf4, double quant,
compute dtype fp16, attn_implementation eager.

실행: 커널 재시작 직후 이 셀만 단독 실행 (baseline 이 CUDA 초기화 전이어야 함).
"""

import subprocess
from collections import Counter

import numpy as np
import torch
from PIL import Image
from transformers import AutoModelForVision2Seq
from transformers import AutoProcessor
from transformers import BitsAndBytesConfig

GB = 1e9  # 기존 기록 (4.38 GB) 과 같은 10진 GB 단위
MIB = 1024**2  # nvidia-smi 출력 단위 (MiB) 환산용


def smi_used_gb():
    """nvidia-smi 로 디바이스 전체 VRAM 사용량을 GB 로 반환한다.

    절대값이 아니라 체크포인트 간 증가분(delta)으로 내 프로세스 몫을
    계산하는 용도 -- 호출부에서 baseline 을 빼서 쓴다.

    Returns:
        디바이스 전체 VRAM 사용량 (10진 GB)
    """
    # 디바이스 전체 사용량 조회 (MiB 숫자만, 헤더/단위 표기 없음)
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout
    return int(out.strip()) * MIB / GB


# --- 시점 0: baseline (반드시 CUDA 초기화 전) ---------------------------------
base = smi_used_gb()  # 다른 프로세스들의 몫 -- 이후 모든 smi 값은 이 값 대비 증가분

# --- 시점 A: CUDA context 단독 ----------------------------------------------
torch.zeros(1, device="cuda")  # 텐서 1개로 CUDA context 초기화 (커널 이미지 로드 포함)
torch.cuda.synchronize()
smi_a = smi_used_gb() - base  # 증가분 = context 단독 크기의 근사 (하한)
print(f"[A] nvidia-smi (context 단독)      : {smi_a:.2f} GB")

# --- 모델 로드 (2026-06 실측과 동일 조건) ------------------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
processor = AutoProcessor.from_pretrained("openvla/openvla-7b", trust_remote_code=True)
vla = AutoModelForVision2Seq.from_pretrained(
    "openvla/openvla-7b",
    attn_implementation="eager",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    quantization_config=bnb_config,
)

# --- 시점 B: 로드 직후 --------------------------------------------------------
torch.cuda.synchronize()
alloc_b = torch.cuda.memory_allocated()  # 살아있는 텐서 바이트 합
reserved_b = torch.cuda.memory_reserved()  # allocator 가 쥔 블록 전체 (allocated 포함)
smi_b = smi_used_gb() - base
print(f"[B] memory_allocated (로드 직후)   : {alloc_b / GB:.2f} GB")
print(f"[B] memory_reserved  (로드 직후)   : {reserved_b / GB:.2f} GB")
print(f"[B] nvidia-smi       (로드 직후)   : {smi_b:.2f} GB")

# --- dtype 별 합산: findings §4.1 Step 3 의 근거 ------------------------------
acc = Counter()
for _, p in vla.named_parameters():
    # int4 가중치는 uint8 에 2개씩 packing 되어 있어 numel x element_size 가 곧 실바이트
    acc[str(p.dtype)] += p.numel() * p.element_size()
for _, b in vla.named_buffers():
    acc["buffer:" + str(b.dtype)] += b.numel() * b.element_size()

# quant_state (absmax scale 등) 는 named_parameters/buffers 에 안 잡히므로 별도 합산
qs_bytes = 0
for m in vla.modules():
    qs = getattr(getattr(m, "weight", None), "quant_state", None)
    if qs is None:
        continue
    tensors = [qs.absmax, qs.code]  # 1차 scale + 코드북
    if qs.state2 is not None:  # double quant 의 2차 scale
        tensors += [qs.state2.absmax, qs.state2.code]
    qs_bytes += sum(t.numel() * t.element_size() for t in tensors)

print("--- dtype 별 합산 (methodology.md §4 표에 붙여넣기) ---")
for dtype, nbytes in sorted(acc.items(), key=lambda kv: -kv[1]):
    print(f"  {dtype:<24}: {nbytes / GB:.3f} GB")
print(f"  {'quant_state (scale 등)':<24}: {qs_bytes / GB:.3f} GB")
total = sum(acc.values()) + qs_bytes
print(
    f"  합계 {total / GB:.2f} GB vs memory_allocated {alloc_b / GB:.2f} GB "
    f"(잔차 {(alloc_b - total) / GB:.3f} GB)"
)

# --- 시점 C: predict_action 1회 후 --------------------------------------------
torch.cuda.reset_peak_memory_stats()  # peak 기준점을 현재 (로드 직후) 로 리셋
image = Image.fromarray((np.random.rand(224, 224, 3) * 255).astype(np.uint8))
prompt = "In: What action should the robot take to pick up the can?\nOut:"
inputs = processor(prompt, image).to("cuda:0", dtype=torch.float16)
# attention_mask 는 전달하지 않는다 (off-by-one 크래시 -- findings.md §3 workaround)
with torch.no_grad():
    vla.predict_action(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        unnorm_key="bridge_orig",
        do_sample=False,
    )
torch.cuda.synchronize()
peak_c = torch.cuda.max_memory_allocated()  # 추론 중 텐서 peak (KV cache·중간 activation 포함)
reserved_c = torch.cuda.memory_reserved()
smi_c = smi_used_gb() - base
print(f"[C] max_memory_allocated (추론 후) : {peak_c / GB:.2f} GB")
print(f"[C] memory_reserved      (추론 후) : {reserved_c / GB:.2f} GB")
print(f"[C] nvidia-smi           (추론 후) : {smi_c:.2f} GB")

# --- 차이 3분해: findings §4.1 Step 4 의 delta --------------------------------
print("--- 차이 3분해 ---")
print(f"  allocator 예약  (B: reserved - allocated)    : {(reserved_b - alloc_b) / GB:.2f} GB")
print(f"  CUDA context    (A 단독 / B: smi - reserved) : {smi_a:.2f} / {smi_b - reserved_b / GB:.2f} GB")
print(f"  activation·KV   (C: peak - B: allocated)     : {(peak_c - alloc_b) / GB:.2f} GB")

/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[A] nvidia-smi (context 단독)      : 0.19 GB


/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  

[B] memory_allocated (로드 직후)   : 4.38 GB
[B] memory_reserved  (로드 직후)   : 4.55 GB
[B] nvidia-smi       (로드 직후)   : 4.74 GB
--- dtype 별 합산 (methodology.md §4 표에 붙여넣기) ---
  torch.uint8             : 3.638 GB
  torch.float16           : 0.531 GB
  buffer:torch.float16    : 0.034 GB
  buffer:torch.float32    : 0.000 GB
  quant_state (scale 등)   : 0.116 GB
  합계 4.32 GB vs memory_allocated 4.38 GB (잔차 0.066 GB)
[C] max_memory_allocated (추론 후) : 4.73 GB
[C] memory_reserved      (추론 후) : 4.84 GB
[C] nvidia-smi           (추론 후) : 5.05 GB
--- 차이 3분해 ---
  allocator 예약  (B: reserved - allocated)    : 0.17 GB
  CUDA context    (A 단독 / B: smi - reserved) : 0.19 / 0.19 GB
  activation·KV   (C: peak - B: allocated)     : 0.34 GB


/workspace/study/physical-ai-study/Studies/Phase 4/.venv-vla/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
